# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is published via a Croissant schema and is accessible from:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant package
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata_obj = dataset.metadata

print(f"{metadata_obj.name}: {metadata_obj.description}")
print("\nAuthors:")
if hasattr(metadata_obj, 'author'):
    for author in metadata_obj.author:
        print(author['@id'])  # Reference authors by their @id

print("\nCitations:")
if hasattr(metadata_obj, 'citation'):
    for citation in metadata_obj.citation:
        print(citation['@id'])

print("\nKeywords:")
if hasattr(metadata_obj, 'keywords'):
    print(', '.join(metadata_obj.keywords))

# Reference dataset by its @id
print(f"\nDataset @id: {metadata_obj['@id']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List all record sets and their field & column IDs

record_sets = []
fields_by_record_set = {}

# Extract record sets from metadata
if hasattr(metadata_obj, 'recordSet') and metadata_obj.recordSet:
    for rs in metadata_obj.recordSet:
        if isinstance(rs, dict):
            # Reference the record set by its @id
            rs_id = rs.get('@id', None)
            if rs_id:
                record_sets.append(rs_id)
                print(f"Record Set @id: {rs_id}")
                # Find the fields belonging to this record set
                if 'field' in rs:
                    fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
                    field_ids = []
                    for f in fields:
                        fid = f.get('@id', str(f))
                        field_ids.append(fid)
                        print(f"  Field @id: {fid}")
                    fields_by_record_set[rs_id] = field_ids
                # Show columns if present
                if 'column' in rs:
                    columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
                    for col in columns:
                        cid = col.get('@id', str(col))
                        print(f"  Column @id: {cid}")
else:
    print('No record sets found in the metadata.')

# If record_sets is empty, try to discover from dataset.records() API
if not record_sets:
    try:
        # Fetch possible record set ids from dataset.records()
        rs_ids = dataset.record_sets()  # API lists possible record_set ids
        record_sets = rs_ids
        print("Discovered record set IDs:")
        for rs_id in rs_ids:
            print(f" - {rs_id}")
    except Exception as e:
        print("Could not discover record sets:", e)

# Preview one or more records for each record set by @id
for rs_id in record_sets:
    print(f"\nSample records for record set @id: {rs_id}")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rs_id)):
            print(json.dumps(rec, indent=2))
            if idx >= 1:
                break
    except Exception as e:
        print(f"Error fetching records for {rs_id}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s referenced above.

In [ ]:
# Extract all available record sets by their @id
# If no record_sets discovered above, set explicitly

if not record_sets:
    record_sets = ['cr:KilifiMentalHealthSurvey']  # Example placeholder, replace if known

dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"\nRecord set {rs_id} - Columns:")
            print(df.columns.tolist())
            print("Preview:")
            print(df.head(2))
        else:
            print(f"No records found for {rs_id}")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}")

# For demonstration, select first record set
selected_record_set = record_sets[0] if record_sets else None
if selected_record_set:
    print(f"\nUsing record set @id: {selected_record_set} for EDA")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, removing outliers, grouping data. All fields referred by `@id`.

In [ ]:
# Select a numeric field by its @id (e.g. GAD-7 score, PHQ-9 score)
# Discover possible numeric fields
numeric_field_id = None
group_field_id = None

if selected_record_set and selected_record_set in dataframes:
    df = dataframes[selected_record_set]
    # Try to find numeric columns (e.g. GAD score, PHQ score)
    numeric_candidates = [col for col in df.columns if 'score' in col or 'GAD' in col or 'PHQ' in col or 'PSQ' in col]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use first match
        print(f"Numeric field selected: {numeric_field_id} (@id)")
    else:
        # Try to find integer/float columns
        numeric_field_id = next((col for col in df.select_dtypes([int, float]).columns), None)
        print(f"Numeric field selected: {numeric_field_id} (@id)")

    # Try to find categorical/grouping field (e.g. demographic)
    possible_groups = ['gender', 'level_of_education', 'marital_status']
    for group_candidate in possible_groups:
        for col in df.columns:
            if group_candidate in col.lower():
                group_field_id = col
                print(f"Group field selected: {group_field_id} (@id)")
                break
        if group_field_id:
            break

    # Filtering based on the numeric field
    if numeric_field_id:
        threshold = 10
        if numeric_field_id in df.columns:
            # Remove outliers above threshold
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())
            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group and aggregate by group_field_id
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped data by {group_field_id} (@id):")
                print(grouped_df.head())
        else:
            print(f"Numeric field {numeric_field_id} not found in columns.")
else:
    print("No valid record set data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing fields and record sets by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if data is available
if selected_record_set and selected_record_set in dataframes and numeric_field_id:
    df = dataframes[selected_record_set]
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    df[numeric_field_id].dropna().hist(bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    # Boxplot by grouping field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to explore the FAIR² Mental Health Survey dataset using the `mlcroissant` library.
- Data was loaded and referenced by unique Croissant `@id`s.
- Record sets and fields were accessed and previewed dynamically.
- Common EDA steps (filtering, normalization, grouping) used field `@id`s for clarity and reproducibility.
- Visualizations highlighted key numeric indicators and demographic grouping.

Further analysis could include deeper statistical assessment or supervised learning, again referencing fields and subsets via `@id` for FAIR/AI-ready workflows.